# YOLOv11 학습 노트북 — Lost-and-Found

**데이터 검증 결과 요약**
- 이미지 512×512 고정 (Roboflow 전처리)
- Roboflow 이미 rotation/brightness/blur 3× 증강 완료
- 클래스 불균형 846× (Phone 5075 vs Charging-cable 6)
- Umbrella: valid/test 샘플 없음

**실행 순서**
1. 환경 설정
2. **테스트 학습** — 소량 데이터, 적은 epoch (동작 확인용)
3. **전체 학습** — 전체 데이터, 본격 학습

---
## 데이터셋 다운로드
> Google Drive에서 데이터셋을 받아옵니다. 이미 폴더가 있으면 건너뜁니다.

In [ ]:
from pathlib import Path
import zipfile

# gdown 없으면 설치
try:
    import gdown
except ImportError:
    import subprocess; subprocess.run(['pip', 'install', 'gdown', '-q'])
    import gdown

ROOT = Path('.').resolve()

DATASETS = {
    'dataset_mini':     '13rstUUK-5GgVirwBNLIeYEsw5xq2vFCS',
    'dataset_balanced': '1pYOy2JVQWsitxZDfN1pg0iWZkWKjpbn9',
}

for name, file_id in DATASETS.items():
    dst = ROOT / name
    if dst.exists():
        print(f'{name} 이미 존재, 건너뜀')
        continue
    zip_path = ROOT / f'{name}.zip'
    print(f'다운로드 중: {name} ...')
    gdown.download(id=file_id, output=str(zip_path), quiet=False)
    print(f'압축 해제 중: {name} ...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(ROOT)
    zip_path.unlink()
    print(f'{name} 완료')

print('\n모든 데이터셋 준비 완료')

---
## 0. 환경 확인 및 전역 설정

In [ ]:
import torch, yaml, shutil, random
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd

# ── 경로 ──────────────────────────────────────────────────
ROOT     = Path(".").resolve()
DATA_DIR = ROOT / "dataset"
DATA_YAML = DATA_DIR / "data.yaml"
RUNS_DIR = ROOT / "runs"

# ── 실험 설정 (여기서만 수정) ──────────────────────────────
# GPU VRAM에 따라 MODEL 선택:
#   ~4GB → yolo11n.pt  /  ~6GB → yolo11s.pt  /  ~10GB → yolo11m.pt
MODEL    = "yolo11s.pt"
EPOCHS   = 100
BATCH    = 16        # VRAM 부족 시 8
EXP_NAME = "full_run_v1"
SEED     = 42

# ── 데이터 메타 ────────────────────────────────────────────
with open(DATA_YAML) as f:
    meta = yaml.safe_load(f)
CLASS_NAMES  = meta["names"]
NC           = meta["nc"]
DEVICE       = 0 if torch.cuda.is_available() else "cpu"
TRAIN_COUNTS = [717, 6, 15, 3882, 144, 15, 5075, 1368, 117, 681, 591, 9]

# ── 헬퍼 함수 ──────────────────────────────────────────────
def make_axes(n, figsize):
    """n개 이미지용 axes 리스트 반환 (n=1일 때 리스트로 정규화)."""
    fig, axes = plt.subplots(1, n, figsize=figsize)
    return fig, [axes] if n == 1 else list(axes)

def show_image_grid(paths, title=None, figsize_per=(5, 4)):
    """이미지 경로 리스트를 한 행으로 표시."""
    if not paths:
        return
    fig, axes = make_axes(len(paths), (figsize_per[0] * len(paths), figsize_per[1]))
    for ax, p in zip(axes, paths):
        ax.imshow(mpimg.imread(p))
        ax.set_title(Path(p).name, fontsize=8)
        ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

def print_cfg(cfg):
    for k, v in cfg.items():
        print(f"  {k:<20} = {v}")

# ── 환경 출력 ──────────────────────────────────────────────
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"\n모델    : {MODEL}")
print(f"클래스  : {CLASS_NAMES}")
print(f"데이터  : {DATA_YAML}")

---
## 1. 테스트 학습
> **목적**: 코드 동작 확인, 하이퍼파라미터 감 잡기  
> 전체 데이터 중 일부만 샘플링한 미니 데이터셋으로 5 epoch만 학습

- 오류 없이 끝나면 → **섹션 2 전체 학습**으로 이동

In [ ]:
# ── 테스트용 미니 데이터셋 생성 ────────────────────────────
MINI_DIR   = ROOT / "dataset_mini"
MINI_SIZES = {"train": 200, "valid": 50, "test": 30}

random.seed(SEED)

def copy_split(src_split: Path, dst_split: Path, n: int) -> int:
    for sub in ("images", "labels"):
        (dst_split / sub).mkdir(parents=True, exist_ok=True)

    all_imgs = sorted(src_split.glob("images/*.jpg")) + \
               sorted(src_split.glob("images/*.png"))
    sampled  = random.sample(all_imgs, min(n, len(all_imgs)))

    for img_path in sampled:
        shutil.copy(img_path, dst_split / "images" / img_path.name)
        lbl = src_split / "labels" / (img_path.stem + ".txt")
        if lbl.exists():
            shutil.copy(lbl, dst_split / "labels" / lbl.name)
    return len(sampled)

if MINI_DIR.exists():
    shutil.rmtree(MINI_DIR)

counts = {split: copy_split(DATA_DIR / split, MINI_DIR / split, n)
          for split, n in MINI_SIZES.items()}

mini_yaml_path = MINI_DIR / "data.yaml"
with open(mini_yaml_path, "w") as f:
    yaml.dump({
        "train": str(MINI_DIR / "train" / "images"),
        "val"  : str(MINI_DIR / "valid" / "images"),
        "test" : str(MINI_DIR / "test"  / "images"),
        "nc"   : NC,
        "names": CLASS_NAMES,
    }, f, allow_unicode=True, default_flow_style=False)

print("미니 데이터셋 생성 완료")
for split, cnt in counts.items():
    print(f"  {split}: {cnt}장")
print(f"  yaml: {mini_yaml_path}")

In [ ]:
# ── 테스트 학습 설정 ───────────────────────────────────────
TEST_CFG = dict(
    data          = str(mini_yaml_path),
    imgsz         = 512,
    epochs        = 5,
    batch         = 8,
    workers       = 2,
    device        = DEVICE,
    project       = str(RUNS_DIR),
    name          = "test_run",
    exist_ok      = True,
    optimizer     = "AdamW",
    lr0           = 0.001,
    warmup_epochs = 1,
    mosaic        = 1.0,
    plots         = True,
    verbose       = True,
    val           = True,
)
print("테스트 학습 설정:")
print_cfg(TEST_CFG)

In [ ]:
# ── 테스트 학습 실행 ───────────────────────────────────────
test_model   = YOLO(MODEL)
test_results = test_model.train(**TEST_CFG)
test_save_dir = Path(test_results.save_dir)

print(f"\n테스트 학습 완료: {test_save_dir}")

In [ ]:
# ── 테스트 결과 확인 ───────────────────────────────────────
results_png = test_save_dir / "results.png"
if results_png.exists():
    plt.figure(figsize=(16, 6))
    plt.imshow(mpimg.imread(results_png))
    plt.axis("off")
    plt.title("테스트 학습 결과 곡선", fontsize=14)
    plt.tight_layout()
    plt.show()

show_image_grid(sorted(test_save_dir.glob("val_batch*.jpg"))[:2],
                title="Validation 예측 샘플")

# train() 이 이미 val을 실행했으므로 results_dict에서 바로 읽음
rd = test_results.results_dict
print(f"\n[테스트 학습 val 지표]")
print(f"  mAP50   : {rd.get('metrics/mAP50(B)', 'N/A')}")
print(f"  mAP50-95: {rd.get('metrics/mAP50-95(B)', 'N/A')}")
print("\n→ 오류 없이 완료됐으면 아래 전체 학습 섹션으로 이동하세요.")

---
## 2. 전체 학습

> 테스트 학습에서 오류 없음을 확인한 후 실행

**모델/VRAM/BATCH는 섹션 0의 전역 설정에서 변경**

In [ ]:
# ── 클래스 불균형 현황 출력 ────────────────────────────────
print("=" * 55)
print("클래스 불균형 현황 (train bbox 기준)")
print("-" * 55)
for name, cnt in zip(CLASS_NAMES, TRAIN_COUNTS):
    bar = "█" * int(cnt / max(TRAIN_COUNTS) * 25)
    print(f"  {name:<18} {cnt:>5}  {bar}")
mx, mn = max(TRAIN_COUNTS), min(TRAIN_COUNTS)
print(f"\n  불균형 비율 : {mx/mn:.0f}x  (max={mx}, min={mn})")
print("  [주의] Umbrella: valid/test 0개 → mAP 미표시")
print("  [주의] Charging-cable/iPad/Earphones/Pen 극히 적음")
print("=" * 55)

In [ ]:
# ── 전체 학습 설정 ─────────────────────────────────────────
FULL_CFG = dict(
    data    = str(DATA_YAML),
    imgsz   = 512,
    epochs  = EPOCHS,
    batch   = BATCH,
    workers = 4,
    device  = DEVICE,
    project = str(RUNS_DIR),
    name    = EXP_NAME,
    exist_ok= False,

    # ─ 옵티마이저
    optimizer    = "AdamW",
    lr0          = 0.001,
    lrf          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    warmup_epochs    = 3,
    warmup_momentum  = 0.8,

    # ─ 손실 가중치
    box = 7.5,
    cls = 0.5,
    dfl = 1.5,

    # ─ 증강 (Roboflow: rotation±15°, brightness±17%, blur 이미 적용)
    degrees   = 0.0,   # Roboflow에서 이미 적용
    hsv_h     = 0.015,
    hsv_s     = 0.5,
    hsv_v     = 0.3,   # Roboflow brightness와 중복 방지
    translate = 0.1,
    scale     = 0.5,
    shear     = 0.0,
    flipud    = 0.0,
    fliplr    = 0.5,
    mosaic    = 1.0,
    mixup     = 0.1,
    copy_paste= 0.1,
    erasing   = 0.4,

    # ─ 학습 제어
    patience    = 20,
    save_period = 10,
    val         = True,
    plots       = True,
    verbose     = True,
)

print("전체 학습 설정:")
print_cfg(FULL_CFG)

In [ ]:
# ── 전체 학습 실행 ─────────────────────────────────────────
full_model   = YOLO(MODEL)
full_results = full_model.train(**FULL_CFG)
full_save_dir = Path(full_results.save_dir)

print(f"\n학습 완료")
print(f"결과 저장 위치 : {full_save_dir}")
print(f"최적 가중치    : {full_save_dir / 'weights' / 'best.pt'}")

---
## 3. 학습 결과 분석

In [ ]:
# ── 학습 곡선 시각화 ───────────────────────────────────────
results_png = full_save_dir / "results.png"
if results_png.exists():
    plt.figure(figsize=(18, 8))
    plt.imshow(mpimg.imread(results_png))
    plt.axis("off")
    plt.title("학습 곡선 (Loss / mAP)", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── CSV에서 직접 그래프 ────────────────────────────────────
csv_path = full_save_dir / "results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.flatten()

    metrics = [
        ("train/box_loss",      "Train Box Loss"),
        ("train/cls_loss",      "Train Cls Loss"),
        ("val/box_loss",        "Val Box Loss"),
        ("val/cls_loss",        "Val Cls Loss"),
        ("metrics/mAP50(B)",    "mAP@50"),
        ("metrics/mAP50-95(B)", "mAP@50-95"),
    ]
    for ax, (col, title) in zip(axes, metrics):
        if col in df.columns:
            ax.plot(df["epoch"], df[col], linewidth=2)
            ax.set_title(title, fontweight="bold")
            ax.set_xlabel("Epoch")
            ax.grid(alpha=0.3)
        else:
            ax.text(0.5, 0.5, f"{col}\n없음", ha="center", va="center",
                    transform=ax.transAxes)
            ax.axis("off")

    plt.suptitle("학습 지표 상세", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Test set 최종 평가 ─────────────────────────────────────
best_model   = YOLO(str(full_save_dir / "weights" / "best.pt"))
test_metrics = best_model.val(
    data=str(DATA_YAML), split="test", imgsz=512, batch=BATCH, save_json=True
)

print("\n" + "=" * 50)
print("Test Set 최종 평가")
print("=" * 50)
print(f"  mAP@50      : {test_metrics.box.map50:.4f}")
print(f"  mAP@50-95   : {test_metrics.box.map:.4f}")
print(f"  Precision   : {test_metrics.box.mp:.4f}")
print(f"  Recall      : {test_metrics.box.mr:.4f}")
print("\n클래스별 AP@50:")
for name, ap in zip(CLASS_NAMES, test_metrics.box.ap50):
    print(f"  {name:<18} {ap:.4f}  {'█' * int(ap * 20)}")
print("=" * 50)

In [ ]:
# ── Confusion Matrix / PR / F1 곡선 ───────────────────────
show_files = [
    ("confusion_matrix_normalized.png", "Confusion Matrix"),
    ("PR_curve.png",                    "PR Curve"),
    ("F1_curve.png",                    "F1 Curve"),
]
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (fname, title) in zip(axes, show_files):
    p = full_save_dir / fname
    if p.exists():
        ax.imshow(mpimg.imread(p))
        ax.set_title(title, fontsize=11, fontweight="bold")
    else:
        ax.text(0.5, 0.5, f"{fname}\n없음", ha="center", va="center",
                transform=ax.transAxes)
    ax.axis("off")
plt.tight_layout()
plt.show()

show_image_grid(
    sorted(full_save_dir.glob("val_batch*pred*.jpg"))[:3],
    title="Validation Predictions"
)

---
## 4. 이어서 학습 (Resume)
> 학습이 중간에 끊겼을 때 재개

In [ ]:
# ── 이어서 학습 ────────────────────────────────────────────
# EXP_NAME은 섹션 0에서 설정. last.pt가 있으면 자동으로 이어서 학습.
LAST_PT = RUNS_DIR / EXP_NAME / "weights" / "last.pt"

if LAST_PT.exists():
    resume_results = YOLO(str(LAST_PT)).train(resume=True)
    print(f"이어서 학습 완료: {resume_results.save_dir}")
else:
    print(f"last.pt 없음: {LAST_PT}")
    print("섹션 0의 EXP_NAME 확인 또는 전체 학습을 먼저 실행하세요.")